## 1. 텍스트 임베딩의 개념

청킹을 통해 문서를 작은 단위로 분할하는 방법을 학습했습니다. 이제 이 청크들을 **검색 가능한 형태** 로 만들어야 합니다.

### 임베딩이란?

**임베딩(Embedding)** 은 텍스트를 **고차원 수치 벡터** 로 변환하는 과정입니다. 변환된 벡터는 텍스트의 **의미적 정보** 를 수치로 인코딩합니다.

```
텍스트                          벡터 (고차원)
──────────────────────────────────────────────────
"인공지능은 미래 기술이다"  →  [0.023, -0.015, 0.078, ..., 0.041]  (1536차원)
"AI는 혁신적 기술이다"     →  [0.021, -0.012, 0.075, ..., 0.039]  (유사한 벡터)
"오늘 날씨가 좋다"         →  [-0.045, 0.032, -0.018, ..., 0.063] (다른 벡터)
```

### 핵심 특성

- **의미적으로 유사한 텍스트** 는 벡터 공간에서 **가까이** 위치한다
- **의미적으로 다른 텍스트** 는 벡터 공간에서 **멀리** 위치한다
- 이를 통해 **키워드 매칭이 아닌 의미 기반 검색** 이 가능해진다

## 2. OpenAI Embedding API 활용

OpenAI의 `text-embedding-3-small` 모델을 사용하여 텍스트를 벡터로 변환합니다.

In [3]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def get_embedding(text, model = 'text-embedding-3-small'):
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

# 다양한 주제의 텍스트로 임베딩 테스트
texts = [
    "인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.",
    "오늘 서울의 날씨는 맑고 기온은 25도이다.",
    "파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.",
]

for text in texts:
    print(f'\n테스트 : {text}')
    print(f'벡터차원 : {len(get_embedding(text))}')
    print(f'벡터 앞 5개 값을 : {get_embedding(text)[:5]}')





테스트 : 인공지능은 인간의 학습 능력을 컴퓨터로 구현한 기술이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.0260009765625, 0.0192413330078125, -0.03167724609375, 0.01084136962890625, 0.0200042724609375]

테스트 : 머신러닝은 데이터로부터 패턴을 학습하는 AI의 한 분야이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [-0.03070068359375, 0.0027484893798828125, 0.027984619140625, 0.0135040283203125, 0.069580078125]

테스트 : 오늘 서울의 날씨는 맑고 기온은 25도이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.018035888671875, -0.01081085205078125, -0.06939697265625, -0.036651611328125, 0.0235443115234375]

테스트 : 파이썬은 데이터 과학에 널리 사용되는 프로그래밍 언어이다.
벡터차원 : 1536
벡터 앞 5개 값을 : [0.0020313262939453125, 0.06939697265625, -0.01097869873046875, -0.0006170272827148438, 0.07879638671875]


## 3. 코사인 유사도와 벡터 검색

두 벡터 간의 **유사도**를 측정하는 대표적인 방법이 **코사인 유사도(Cosine Similarity)** 입니다. 값이 1에 가까울수록 유사하고, 0에 가까울수록 무관합니다.

In [ ]:
embeddings = [ get_embedding(t) for t in texts]
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a,b)/(np.linalg.norm(a) * np.linalg.norm(b)))

# 헤더 출력
print(f"{'':>8}", end="")
for j in range(len(texts)):
    print(f"텍스트{j+1:>2}", end="  ")
print()

for i in range(len(texts)):
    print(f"텍스트{i+1}", end=" ")
    for j in range(len(texts)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"  {sim:.4f}", end="")
    print()

print()
print("[텍스트 목록]")
for i, t in enumerate(texts):
    print(f"  텍스트{i+1}: {t}")

print()
print("[유사도 해석]")
sim_12 = cosine_similarity(embeddings[0], embeddings[1])
sim_13 = cosine_similarity(embeddings[0], embeddings[2])

print(f"  AI-머신러닝 유사도: {sim_12:.4f}")
print(f"  AI-날씨 유사도:     {sim_13:.4f}")